In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from deap import base, creator, tools, algorithms
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression, SelectKBest
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

station='sf'#Specify the station name by choosing the appropriate folder sheet name. In this case, it is 'sf' standing for San Francisco. you can choose among [sf,mi,ph,ch]
file_path = f".\Sample Date\historical Inputs\station_{station}\PS_inputs.xlsx"
data = pd.read_excel(file_path)
x=data.iloc[:, :-2]
X = data.iloc[:, :-2].values
y = data['prism'].values

In [ ]:
real_list = np.unique([i[:-1] for i in x.columns])
for r in real_list:
    for col in x.columns:
        if r in col:
            x[f'{r}_avg'] = np.mean(x[[f'{r}1', f'{r}2', f'{r}3', f'{r}4']].values, axis=1)
avg_list = [i for i in x.columns if 'avg' in i]
x['prism'] = y  # Ensure 'prism' is included in the dataframe
x = x[avg_list]


In [ ]:


feature_names_list = [f"{i}" for i in x.columns]
X_df = pd.DataFrame(x, columns=feature_names_list)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
print("Explained variance ratio:", pca.explained_variance_ratio_)

feature_names = X_df.columns
components_df = pd.DataFrame(pca.components_, columns=feature_names)
print("PCA Components (Loadings):")
print(components_df)

for i, component in enumerate(components_df.values):
    print(f"Principal Component {i+1}:")
    sorted_idx = component.argsort()[::-1]
    sorted_features = feature_names[sorted_idx]
    print(f"  Top features: {sorted_features[:5]}")


In [ ]:


# Ensure there are no missing values in the data
x = x.fillna(0)
y = np.nan_to_num(y)

# Check the mutual information scores directly
mi_selector = SelectKBest(mutual_info_regression, k=3)
X_mi_selected = mi_selector.fit_transform(x, y)
selected_indices = mi_selector.get_support(indices=True)

if isinstance(x, pd.DataFrame):
    selected_feature_names = x.columns[selected_indices]
else:
    selected_feature_names = [f"Feature_{i}" for i in selected_indices]
print("Selected features by MI:", selected_feature_names)

mi_scores = mi_selector.scores_
print("Mutual Information Scores Mean:", np.mean(mi_scores))
print("Mutual Information Scores Mean:", mi_scores)

plt.figure(figsize=(10, 6))
plt.bar(x.columns, mi_scores)
plt.plot(np.arange(len(x.columns)), [np.mean(mi_scores)] * len(x.columns), color='red', label='Mean MI Score')
plt.xlabel('Features')
plt.ylabel('MI Scores')
plt.title('Mutual Information Scores of Features')
plt.xticks(rotation=90)
plt.show()

In [ ]:

model = DecisionTreeRegressor(random_state=42)
model.fit(x, y)
importances = model.feature_importances_

top_k = 5
selected_indices = np.argsort(importances)[-top_k:]

if isinstance(x, pd.DataFrame):
    selected_feature_names = x.columns[selected_indices]
else:
    selected_feature_names = [f"Feature_{i}" for i in selected_indices]
print("Selected features based on Decision Tree importance:", selected_feature_names)
